# Сравнение точного и приближённого поиска соседей на MNIST

Цель — сравнить **nearest-neighbor retrieval**: Exact Brute Force (единственный ground truth),
FlyHash, HNSW и IVFPQ. Сравниваются индексы соседей, а не классы цифр.
Метрики: Recall@1/5/10, построение, поиск и дополнительный пиковый RSS.

Запускайте ячейки сверху вниз в Python 3.11–3.12. Рабочая директория должна содержать
`data/mnist_784.csv`. Данные не скачиваются автоматически и не заменяются синтетическими.
При отсутствии CSV выполнение останавливается с понятным сообщением.
Результаты появятся в `results/` только после выполнения соответствующих этапов.

## 1. Установка зависимостей

Первая техническая ячейка устанавливает через `%pip` только отсутствующие или отличающиеся
от проверенных версий сторонние пакеты. Стандартные библиотеки не устанавливаются.
Для `FlyHash 0.3.0` требуется NumPy < 2; используем NumPy 1.26.4.
Версии закреплены для воспроизводимости. Для сборки `hnswlib` может понадобиться C++11-компилятор.
Если выбранный Python-компилятор отсутствует, а GCC доступен, используется GCC.
Если уже импортированные пакеты обновлены, потребуется перезапуск ядра и запуск сверху вниз.
Недоступный FlyHash вызывает явную ошибку, другой алгоритм вместо него не используется.

In [ ]:
import importlib.metadata as metadata
import os
import shlex
import shutil
import sys
import sysconfig

if not (3, 11) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("Закреплённые зависимости рассчитаны на Python 3.11–3.12.")

DEPENDENCIES = {
    "numpy": ("1.26.4", "numpy"),
    "pandas": ("2.2.3", "pandas"),
    "matplotlib": ("3.10.8", "matplotlib"),
    "scikit-learn": ("1.8.0", "sklearn"),
    "psutil": ("7.2.2", "psutil"),
    "threadpoolctl": ("3.6.0", "threadpoolctl"),
    "flyhash": ("0.3.0", "flyhash"),
    "hnswlib": ("0.8.0", "hnswlib"),
    "faiss-cpu": ("1.15.1", "faiss"),
}
missing = []
loaded_changed = []
for package, (expected, module) in DEPENDENCIES.items():
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != expected:
        missing.append(f"{package}=={expected}")
        if module in sys.modules:
            loaded_changed.append(module)

if missing:
    configured_cxx = shlex.split(sysconfig.get_config_var("CXX") or "c++")[0]
    if ("CXX" not in os.environ and shutil.which(configured_cxx) is None
            and shutil.which("g++") and shutil.which("gcc")):
        os.environ["CC"] = "gcc"
        os.environ["CXX"] = "g++"
        print("Для сборки расширений выбран доступный GCC.")
    package_arguments = " ".join(shlex.quote(item) for item in missing)
    print("Установка зависимостей:", package_arguments)
    %pip install --disable-pip-version-check --no-input {package_arguments}
    unavailable = []
    for package, (expected, _) in DEPENDENCIES.items():
        try:
            installed = metadata.version(package)
        except metadata.PackageNotFoundError:
            installed = None
        if installed != expected:
            unavailable.append(f"{package}=={expected}")
    if unavailable:
        raise RuntimeError(
            "Установка не завершена: " + ", ".join(unavailable)
            + ". Проверьте вывод pip, доступ к реестру и C++-компилятор. "
            + "FlyHash нельзя заменять собственной случайной проекцией."
        )
    if loaded_changed:
        raise RuntimeError("Перезапустите ядро и выполните notebook сверху вниз: обновлены "
                           + ", ".join(loaded_changed))
else:
    print("Все необходимые зависимости установлены в закреплённых версиях.")

## 2. Импорты

Импортируются реальные библиотеки алгоритмов. Неудачный импорт не скрывается.
Кэш Matplotlib помещается в доступную временную директорию.

In [ ]:
from pathlib import Path
import gc
import hashlib
import inspect
import json
import pickle
import platform
import random
import tempfile
import threading
import time
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "ann_mnist_mpl"))
os.environ.setdefault("XDG_CACHE_HOME", str(Path(tempfile.gettempdir()) / "ann_mnist_cache"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from threadpoolctl import threadpool_limits, threadpool_info
from IPython.display import display, Markdown

try:
    from flyhash import FlyHash
except ImportError as exc:
    raise ImportError("FlyHash недоступен после установки. Этап FlyHash нельзя выполнить.") from exc
import hnswlib
import faiss

VERSIONS = {name: metadata.version(name) for name in DEPENDENCIES}
VERSIONS["scipy"] = metadata.version("scipy")  # Транзитивная зависимость scikit-learn.
print("Версии библиотек:")
display(pd.DataFrame(VERSIONS.items(), columns=["Библиотека", "Версия"]))
print("Реальный API FlyHash:", inspect.signature(FlyHash))
required_arguments = {"input_dim", "hash_dim", "sparsity", "seed", "quant_step"}
if not required_arguments.issubset(inspect.signature(FlyHash).parameters):
    raise RuntimeError("API FlyHash отличается от проверенного API 0.3.0; требуется адаптация.")

## 3. Конфигурация и технические решения

Все методы используют один split: **10 000 train / 500 test**, 784 признака, K=10, seed=42.
FlyHash: hash_dimension=7840, sparsity_ratio=0.05; плотность проекции остаётся библиотечной
0.1, `quant_step=None` даёт бинарные коды. HNSW: L2, M=16 (стандартное значение),
ef_construction=200, ef=50. IVFPQ: L2, nlist=100, m=28, bits=8, nprobe=10.

Не определённые в задании вычислительные настройки: один поток у всех методов и фиксированный
порядок Exact → FlyHash → HNSW → IVFPQ. Это уменьшает недетерминизм и избегает переподписки CPU.
Один измеряемый пакет из 500 запросов, без подбора параметров и без скрытого прогрева.
Для кодирования FlyHash используются блоки по 128 строк; Hamming считается по одной query
и блокам train по 1024 строки. Эти размеры меняют расход временной памяти, а не алгоритм.
Равные Hamming distances сортируются по индексу train. Окружение и split сохраняются.
Время и RSS могут отличаться между запусками даже при одинаковых соседях.

In [ ]:
RANDOM_STATE = 42
N_TRAIN = 10000
N_TEST = 500
K_MAX = 10
DATA_PATH = Path("data/mnist_784.csv")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

D = 784
HASH_DIMENSION = 7840
SPARSITY_RATIO = 0.05
HNSW_M = 16
EF_CONSTRUCTION = 200
EF = 50
NLIST = 100
M_SUBQUANTIZERS = 28
BITS_PER_CODE = 8
NPROBE = 10
NUM_THREADS = 1
ENCODE_BATCH_SIZE = 128
HAMMING_TRAIN_BATCH = 1024
RSS_INTERVAL_S = 0.005
MB_BYTES = 1_000_000  # Десятичный мегабайт; одинаково для всех методов.

assert D % M_SUBQUANTIZERS == 0, "Размерность должна делиться на число подквантизаторов."
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
faiss.omp_set_num_threads(NUM_THREADS)
# Объект сохраняется на весь эксперимент: лимит не снимается после выхода из ячейки.
thread_limiter = threadpool_limits(limits=NUM_THREADS)
configuration = {
    "random_state": RANDOM_STATE, "n_train": N_TRAIN, "n_test": N_TEST,
    "k": K_MAX, "dimension": D, "threads": NUM_THREADS,
    "exact": {"algorithm": "brute", "metric": "euclidean", "n_neighbors": K_MAX},
    "flyhash": {"hash_dimension": HASH_DIMENSION, "sparsity_ratio": SPARSITY_RATIO,
                "density": 0.1, "quant_step": None, "seed": RANDOM_STATE},
    "hnsw": {"space": "l2", "M": HNSW_M, "ef_construction": EF_CONSTRUCTION,
             "ef": EF, "random_seed": RANDOM_STATE},
    "ivfpq": {"metric": "L2", "nlist": NLIST, "m": M_SUBQUANTIZERS,
              "bits": BITS_PER_CODE, "nprobe": NPROBE, "seed": RANDOM_STATE},
    "rss_interval_s": RSS_INTERVAL_S, "megabyte_bytes": MB_BYTES,
    "encode_batch_size": ENCODE_BATCH_SIZE, "hamming_train_batch": HAMMING_TRAIN_BATCH,
}
print("Рабочая директория:", Path.cwd())
print("Конфигурация:")
display(configuration)

## 4. Загрузка и проверка MNIST

CSV должен содержать 784 пиксельных признака и столбец меток 0–9.
Проверяются существование, размер, чтение, числовые значения и конечность.
Известные имена target/label/class/digit проверяются по содержимому.
При неизвестном имени допускается только единственный подходящий столбец; при неоднозначности
выдаётся ошибка, а не догадка. Явный последовательный `Unnamed: 0` индекс CSV удаляется
только после проверки, что он точно равен номерам строк. Исходные индексы эксперимента —
позиции строк исходного CSV. NaN, inf и значения вне диапазона не заполняются: это нарушило бы
неоговорённую методику, поэтому выполнение останавливается.

In [ ]:
def load_mnist_csv(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Файл MNIST отсутствует: {path.resolve()}. "
            "Поместите исходный mnist_784.csv в data/ и повторите запуск сверху вниз."
        )
    if path.stat().st_size == 0:
        raise ValueError("Файл MNIST пуст.")
    try:
        frame = pd.read_csv(path)
    except (pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeError, OSError) as exc:
        raise ValueError(f"Не удалось прочитать CSV MNIST: {exc}") from exc
    if frame.empty:
        raise ValueError("CSV не содержит строк данных.")
    if len(frame) < N_TRAIN + N_TEST:
        raise ValueError(f"Нужно не менее {N_TRAIN + N_TEST} строк, получено {len(frame)}.")

    if frame.shape[1] == D + 2:
        unnamed = [col for col in frame if str(col).strip().lower().startswith("unnamed:")]
        removable = []
        for col in unnamed:
            values = pd.to_numeric(frame[col], errors="coerce").to_numpy()
            if np.array_equal(values, np.arange(len(frame))):
                removable.append(col)
        if len(removable) == 1:
            frame = frame.drop(columns=removable)
            print("Удалён проверенный служебный индекс CSV:", removable[0])
    if frame.shape[1] != D + 1:
        raise ValueError(f"Ожидается 784 признака и одна метка; получено {frame.shape[1]} столбцов.")

    def label_values(series):
        values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            return None
        if not np.equal(values, np.floor(values)).all():
            return None
        if not np.array_equal(np.unique(values), np.arange(10)):
            return None
        return values.astype(np.int64)

    known_names = {"target", "label", "class", "digit", "y"}
    named = [col for col in frame if str(col).strip().lower() in known_names]
    if named:
        if len(named) != 1 or label_values(frame[named[0]]) is None:
            raise ValueError("Целевой столбец неоднозначен или не содержит корректные метки 0–9.")
        target_column = named[0]
    else:
        candidates = [col for col in frame if label_values(frame[col]) is not None]
        if len(candidates) != 1:
            raise ValueError("Нельзя однозначно определить метки цифр по содержимому CSV. "
                             "Используйте однозначный столбец target или label.")
        target_column = candidates[0]
    y_values = label_values(frame[target_column])
    try:
        features = frame.drop(columns=target_column).apply(pd.to_numeric, errors="raise")
        x_values = features.to_numpy(dtype=np.float32, copy=True)
    except (ValueError, TypeError) as exc:
        raise ValueError("Пиксельные признаки CSV должны быть числовыми.") from exc
    if x_values.shape != (len(frame), D):
        raise ValueError("После выделения метки требуется ровно 784 признака.")
    if not np.isfinite(x_values).all():
        raise ValueError("В признаках есть NaN или inf. Требуется корректный исходный MNIST.")
    if x_values.min() < 0 or x_values.max() > 255:
        raise ValueError("Исходные пиксели должны находиться в [0, 255].")
    print("Целевой столбец:", target_column)
    print("Загружено строк:", len(frame), "; признаков:", x_values.shape[1])
    return x_values, y_values, str(target_column)

X, y, target_column = load_mnist_csv(DATA_PATH)
original_indices = np.arange(len(X), dtype=np.int64)

## 5. Общая нормализация

Деление всех пикселей на 255 выполняется один раз до эксперимента.
Подготовка данных не входит ни во время построения, ни во время поиска методов.

In [ ]:
X = X / np.float32(255.0)
if not np.isfinite(X).all() or X.min() < 0 or X.max() > 1:
    raise ValueError("Нормализованные признаки должны быть конечными и находиться в [0, 1].")
print("Минимум:", float(X.min()))
print("Максимум:", float(X.max()))
print("Тип:", X.dtype)
print("Размер:", X.shape)

## 6. Стратифицированное формирование Train/Test

Сначала стратифицированно выбираются ровно 10 500 строк, затем они стратифицированно
делятся на 10 000 train и 500 test. Оба шага используют random_state=42.
Оставшиеся строки не используются для поиска или обучения индексов.
Если CSV содержит ровно 10 500 строк, первый шаг оставляет их все: выбор полного набора
не требует случайного split с пустой оставшейся частью. Второй шаг остаётся неизменным.

In [ ]:
if len(original_indices) == N_TRAIN + N_TEST:
    selected_indices = original_indices.copy()
else:
    selected_indices, unused_indices = train_test_split(
        original_indices, train_size=N_TRAIN + N_TEST,
        random_state=RANDOM_STATE, stratify=y
    )
    del unused_indices
train_original_indices, test_original_indices = train_test_split(
    selected_indices, train_size=N_TRAIN, test_size=N_TEST,
    random_state=RANDOM_STATE, stratify=y[selected_indices]
)
X_train = X[train_original_indices]
X_test = X[test_original_indices]
y_train = y[train_original_indices]
y_test = y[test_original_indices]
X_train_f32 = np.ascontiguousarray(X_train, dtype=np.float32)
X_test_f32 = np.ascontiguousarray(X_test, dtype=np.float32)

## 7. Проверка выборок и сохранение происхождения данных

Проверяются размеры, отсутствие пересечения, диапазон признаков и распределение цифр.
Метки используются только для стратификации и этой проверки.
Исходные индексы позволяют воспроизвести соответствие между соседями и строками CSV.
Полный массив признаков освобождается до начала измерений; все алгоритмы используют
одни и те же неизменяемые contiguous float32 train/test.

In [ ]:
assert len(selected_indices) == N_TRAIN + N_TEST, "Неверный размер выбранной выборки."
assert X_train.shape == (N_TRAIN, D), "Неверный размер train."
assert X_test.shape == (N_TEST, D), "Неверный размер test."
assert np.intersect1d(train_original_indices, test_original_indices).size == 0, "Train и test пересекаются."
assert len(np.unique(train_original_indices)) == N_TRAIN, "В train есть повторяющиеся строки."
assert len(np.unique(test_original_indices)) == N_TEST, "В test есть повторяющиеся строки."
assert np.array_equal(np.sort(np.r_[train_original_indices, test_original_indices]),
                      np.sort(selected_indices)), "Split потерял выбранные строки."
for array in (X_train_f32, X_test_f32):
    assert array.dtype == np.float32 and array.flags.c_contiguous
    assert np.isfinite(array).all() and 0 <= array.min() <= array.max() <= 1
    array.setflags(write=False)

counts = pd.DataFrame({
    "Исходная выборка": np.bincount(y, minlength=10),
    "Train": np.bincount(y_train, minlength=10),
    "Test": np.bincount(y_test, minlength=10),
}, index=pd.Index(range(10), name="Цифра"))
display(counts)
display((counts / counts.sum()).rename(columns=lambda c: c + " — доля"))
np.savez_compressed(RESULTS_DIR / "split_indices.npz",
                    train_original_indices=train_original_indices,
                    test_original_indices=test_original_indices,
                    selected_indices=selected_indices)

file_hash = hashlib.sha256()
with DATA_PATH.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        file_hash.update(chunk)
manifest = {
    "configuration": configuration, "versions": VERSIONS,
    "python": platform.python_version(), "platform": platform.platform(),
    "cpu_count": os.cpu_count(), "threadpools": threadpool_info(),
    "dataset_path": str(DATA_PATH), "dataset_sha256": file_hash.hexdigest(),
    "dataset_rows": len(y), "target_column": target_column,
    "train_sha256": hashlib.sha256(X_train_f32.tobytes()).hexdigest(),
    "test_sha256": hashlib.sha256(X_test_f32.tobytes()).hexdigest(),
}
with (RESULTS_DIR / "experiment_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(manifest, stream, ensure_ascii=False, indent=2)
del X, X_train, X_test
_ = gc.collect()
print("Размеры и отсутствие пересечения проверены; один общий split подготовлен.")

## 8. Общие функции: Recall, время, RSS, валидация, сохранение

Для каждой query: Recall@K = |GT[:K] ∩ ANN[:K]| / K; затем среднее по всем 500 query.
Классы цифр в функцию не передаются. Дополнительно проверяются shape, диапазон индексов
и отсутствие повторяющихся соседей в одной строке.

RSS измеряется одним и тем же sampler каждые 5 мс: после gc.collect() фиксируется RSS перед
методом, затем во время build + search наблюдается максимум. **Peak Memory (MB)** —
max(0, максимальный RSS − исходный RSS), MB = 10⁶ байт. Граничные измерения дополняют sampler.
Дополнительные пиковые значения build/search также отсчитываются от общего исходного RSS,
поэтому search peak включает сохраняющийся индекс.

Ограничения: короткий пик может попасть между измерениями; allocator и Python/native libraries
могут не сразу вернуть память ОС. RSS всего процесса зависит от порядка методов и уже
выделенных областей. Это экспериментальная оценка, а не размер индекса; нулевой прирост
возможен при повторном использовании памяти. RSS учитывает native FAISS/hnswlib лучше,
чем один tracemalloc. Предыдущие индексы освобождаются; сохраняются лишь небольшие результаты.
Сам sampler одинаков у всех методов, но вносит небольшие накладные расходы.
Сохранение файлов, Recall и графики выполняются вне измеряемых участков.

In [ ]:
def validate_indices(indices, name, n_queries=N_TEST, n_train=N_TRAIN):
    indices = np.asarray(indices)
    if indices.shape != (n_queries, K_MAX):
        raise ValueError(f"{name}: ожидается shape {(n_queries, K_MAX)}, получено {indices.shape}.")
    if not np.issubdtype(indices.dtype, np.integer):
        raise ValueError(f"{name}: индексы должны быть целыми числами.")
    if np.any(indices < 0) or np.any(indices >= n_train):
        raise ValueError(f"{name}: индексы должны находиться в [0, {n_train}).")
    if np.any(np.diff(np.sort(indices, axis=1), axis=1) == 0):
        raise ValueError(f"{name}: повторяющиеся соседи в одной query.")
    return indices


def recall_at_k(exact_indices, approx_indices, k):
    gt = np.asarray(exact_indices)
    ann = np.asarray(approx_indices)
    if gt.ndim != 2 or ann.ndim != 2 or gt.shape[0] != ann.shape[0] or gt.shape[0] == 0:
        raise ValueError("Recall требует непустые двумерные массивы с одинаковым числом query.")
    if not isinstance(k, (int, np.integer)) or k < 1 or k > min(gt.shape[1], ann.shape[1]):
        raise ValueError("K должен быть положительным и не превышать число соседей.")
    if not np.issubdtype(gt.dtype, np.integer) or not np.issubdtype(ann.dtype, np.integer):
        raise ValueError("Recall рассчитывается по целочисленным индексам соседей.")
    return float(np.mean([
        np.intersect1d(gt_row[:k], ann_row[:k]).size / k
        for gt_row, ann_row in zip(gt, ann)
    ]))


class PeakRSS:
    def __init__(self, interval=RSS_INTERVAL_S):
        self.interval = interval
        self.process = psutil.Process()
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.phase = "build"
        self.error = None

    def sample(self):
        rss = self.process.memory_info().rss
        with self.lock:
            self.peak = max(self.peak, rss)
            self.phase_peaks[self.phase] = max(self.phase_peaks[self.phase], rss)

    def _run(self):
        try:
            while not self.stop_event.wait(self.interval):
                self.sample()
        except (psutil.Error, OSError) as exc:
            self.error = exc
            self.stop_event.set()

    def __enter__(self):
        gc.collect()
        self.before = self.process.memory_info().rss
        self.peak = self.before
        self.phase_peaks = {"build": self.before, "search": self.before}
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        self.sample()
        return self

    def begin_search(self):
        self.sample()
        with self.lock:
            self.phase = "search"
        self.sample()

    def __exit__(self, exc_type, exc_value, traceback):
        try:
            self.sample()
        finally:
            self.stop_event.set()
            self.thread.join()
        if self.error is not None and exc_type is None:
            raise RuntimeError("Ошибка sampler RSS; оценка памяти недостоверна.") from self.error
        return False

    def metrics(self):
        return {
            "Peak Memory (MB)": max(0, self.peak - self.before) / MB_BYTES,
            "Build Peak Memory (MB)": max(0, self.phase_peaks["build"] - self.before) / MB_BYTES,
            "Search Peak Memory (MB)": max(0, self.phase_peaks["search"] - self.before) / MB_BYTES,
            "RSS Before (MB)": self.before / MB_BYTES,
            "Peak RSS (MB)": self.peak / MB_BYTES,
        }


def timed_call(function, *args, **kwargs):
    started = time.perf_counter()
    result = function(*args, **kwargs)
    return result, time.perf_counter() - started


def benchmark(build_function, search_function):
    with PeakRSS() as memory:
        (index, extra), build_wall = timed_call(build_function)
        memory.begin_search()
        (distances, indices), search_seconds = timed_call(search_function, index)
    stats = {
        "Build/Train Time (s)": extra.get("Build/Train Time (s)", build_wall),
        "Search Time Total (s)": search_seconds,
        "Search Time per Query (ms)": search_seconds / N_TEST * 1000,
        **memory.metrics(), **extra,
    }
    del index
    return distances, indices, stats


def save_neighbors(filename, indices, distances=None, parameters=None, timings=None):
    validate_indices(indices, filename)
    payload = {"indices": indices, "k": K_MAX}
    if distances is not None:
        distances = np.asarray(distances)
        if distances.shape != (N_TEST, K_MAX) or not np.isfinite(distances).all():
            raise ValueError(f"{filename}: некорректные расстояния.")
        payload["distances"] = distances
    if parameters is not None:
        payload["parameters"] = parameters
    if timings is not None:
        payload["timings"] = timings
    path = RESULTS_DIR / filename
    with path.open("wb") as stream:
        pickle.dump(payload, stream, protocol=pickle.HIGHEST_PROTOCOL)
    # Загружаем только собственный доверенный pickle, созданный текущим запуском.
    with path.open("rb") as stream:
        restored = pickle.load(stream)
    if set(restored) != set(payload) or restored["k"] != K_MAX:
        raise ValueError(f"{filename}: неверные ключи или K после загрузки.")
    validate_indices(restored["indices"], filename)
    if not np.array_equal(restored["indices"], indices):
        raise ValueError(f"{filename}: индексы изменились после сохранения.")
    if distances is not None:
        if restored["distances"].shape != (N_TEST, K_MAX):
            raise ValueError(f"{filename}: неверный shape расстояний после загрузки.")
        if not np.array_equal(restored["distances"], distances):
            raise ValueError(f"{filename}: расстояния изменились после сохранения.")
    print("Сохранение и обратная загрузка проверены:", path)


all_indices = {}
all_stats = {}

## 9. Exact Brute Force — ground truth

`NearestNeighbors(n_neighbors=10, algorithm="brute", metric="euclidean")` обучается только
на общем train. Build = fit; search = kneighbors общего test. Exact Recall=1 означает
сравнение ground truth с самим собой, а не качество приближённого алгоритма.
Возвращаются евклидовы расстояния и локальные индексы строк train.

In [ ]:
def build_exact():
    model = NearestNeighbors(n_neighbors=10, algorithm="brute", metric="euclidean")
    _, fit_seconds = timed_call(model.fit, X_train_f32)
    return model, {"Build/Train Time (s)": fit_seconds}


def search_exact(model):
    return model.kneighbors(X_test_f32)


exact_distances, exact_indices, exact_stats = benchmark(build_exact, search_exact)
validate_indices(exact_indices, "Exact")
assert exact_distances.shape == (N_TEST, K_MAX), "Некорректные exact distances."
assert np.isfinite(exact_distances).all() and (exact_distances >= 0).all()
assert (np.diff(exact_distances, axis=1) >= -1e-6).all(), "Exact distances не отсортированы."
for k in (1, 5, 10):
    assert np.isclose(recall_at_k(exact_indices, exact_indices, k), 1.0), "Ошибка sanity check Recall."
save_neighbors("exact_neighbors_k10.pkl", exact_indices, exact_distances,
               configuration["exact"], exact_stats)
all_indices["Exact Brute Force"] = exact_indices
all_stats["Exact Brute Force"] = exact_stats
print("Ground truth рассчитан:", exact_indices.shape)

## 10. FlyHash: бинарное кодирование и Hamming retrieval

Проверенный API 0.3.0: `FlyHash(input_dim=784, hash_dim=7840, sparsity=0.05,
quant_step=None, seed=42)`; hasher является вызываемым объектом, fit не нужен.
Один объект кодирует train и test. Коды проверяются как бинарные и пакуются через packbits.
Библиотечная плотная проекция и её вычисления сохраняются; собственная проекция не подменяет FlyHash.

Hamming — число отличающихся исходных битов, а не байтов: XOR + таблица popcount.
В памяти находится только блок 1024 × 980 байт, а не tensor 500 × 10000 × 7840.
После вычисления 10 000 расстояний применяется стабильная сортировка: порядок train
изначально возрастающий, поэтому ties разрешаются по train_index.
Build включает hasher, train encoding и popcount-структуру; search включает test encoding
тем же hasher и поиск top-10. Это возврат индексов соседей, без обращения к y_train.

In [ ]:
def encode_packed(hasher, data, batch_size=ENCODE_BATCH_SIZE):
    packed = np.empty((len(data), (HASH_DIMENSION + 7) // 8), dtype=np.uint8)
    for start in range(0, len(data), batch_size):
        stop = min(start + batch_size, len(data))
        codes = np.asarray(hasher(data[start:stop]))
        # API squeeze() убирает batch axis для единственной строки.
        codes = codes.reshape(stop - start, HASH_DIMENSION)
        if not np.all((codes == 0) | (codes == 1)):
            raise ValueError("FlyHash вернул небинарные коды; Hamming по битам некорректен.")
        if not np.all(codes.sum(axis=1) == round(HASH_DIMENSION * SPARSITY_RATIO)):
            raise ValueError("FlyHash нарушил заданную разреженность бинарных кодов.")
        packed[start:stop] = np.packbits(codes.astype(np.uint8, copy=False), axis=1)
    return packed


def hamming_top_k(train_packed, query_packed, popcount, k=K_MAX):
    indices = np.empty((len(query_packed), k), dtype=np.int64)
    distances = np.empty((len(query_packed), k), dtype=np.int32)
    if len(train_packed) < k or train_packed.shape[1] != query_packed.shape[1]:
        raise ValueError("Недостаточно train-кодов или несовместимая размерность Hamming.")
    for query_id, query_code in enumerate(query_packed):
        all_distances = np.empty(len(train_packed), dtype=np.int32)
        for start in range(0, len(train_packed), HAMMING_TRAIN_BATCH):
            stop = min(start + HAMMING_TRAIN_BATCH, len(train_packed))
            xor = np.bitwise_xor(train_packed[start:stop], query_code)
            all_distances[start:stop] = popcount[xor].sum(axis=1, dtype=np.int32)
        order = np.argsort(all_distances, kind="stable")[:k]
        indices[query_id] = order
        distances[query_id] = all_distances[order]
    return distances, indices


def build_flyhash():
    hasher = FlyHash(input_dim=D, hash_dim=HASH_DIMENSION,
                     sparsity=SPARSITY_RATIO, quant_step=None, seed=RANDOM_STATE)
    train_packed = encode_packed(hasher, X_train_f32)
    popcount = np.array([value.bit_count() for value in range(256)], dtype=np.uint8)
    return (hasher, train_packed, popcount), {}


def search_flyhash(state):
    hasher, train_packed, popcount = state
    test_packed = encode_packed(hasher, X_test_f32)
    return hamming_top_k(train_packed, test_packed, popcount)


flyhash_distances, flyhash_indices, flyhash_stats = benchmark(build_flyhash, search_flyhash)
validate_indices(flyhash_indices, "FlyHash")
assert (np.diff(flyhash_distances, axis=1) >= 0).all(), "Hamming distances не отсортированы."
save_neighbors("flyhash_neighbors_k10.pkl", flyhash_indices, flyhash_distances,
               configuration["flyhash"], flyhash_stats)
all_indices["FlyHash"] = flyhash_indices
all_stats["FlyHash"] = flyhash_stats
print("FlyHash retrieval завершён:", flyhash_indices.shape)

## 11. HNSW

Настоящий hnswlib: space=l2, dim=784, max_elements=10000, ef_construction=200,
M=16, random_seed=42 и один поток. Метки индекса — np.arange(10000), поэтому результат
прямо обозначает строки общего train. ef=50 остаётся неизменным при проверке H1.
Build включает создание, init_index, добавление и настройки; search — только knn_query.
Расстояния hnswlib — квадраты L2, что не меняет порядок ближайших соседей.

In [ ]:
def build_hnsw():
    index = hnswlib.Index(space="l2", dim=D)
    index.init_index(max_elements=N_TRAIN, ef_construction=EF_CONSTRUCTION,
                     M=HNSW_M, random_seed=RANDOM_STATE)
    index.set_num_threads(NUM_THREADS)
    labels = np.arange(N_TRAIN)
    index.add_items(X_train_f32, labels)
    index.set_ef(EF)
    if index.get_current_count() != N_TRAIN:
        raise ValueError("HNSW не содержит все train-объекты.")
    if not np.array_equal(np.sort(np.asarray(index.get_ids_list())), labels):
        raise ValueError("Метки HNSW не совпадают с индексами строк train.")
    return index, {}


def search_hnsw(index):
    labels, squared_distances = index.knn_query(X_test_f32, k=K_MAX)
    return squared_distances, labels.astype(np.int64, copy=False)


hnsw_distances, hnsw_indices, hnsw_stats = benchmark(build_hnsw, search_hnsw)
validate_indices(hnsw_indices, "HNSW")
save_neighbors("hnsw_neighbors_k10.pkl", hnsw_indices, hnsw_distances,
               configuration["hnsw"], hnsw_stats)
all_indices["HNSW"] = hnsw_indices
all_stats["HNSW"] = hnsw_stats
print("HNSW retrieval завершён:", hnsw_indices.shape)

## 12. IVFPQ

Настоящий FAISS IndexIVFPQ с L2-квантизатором: nlist=100, m=28, bits=8, nprobe=10;
784 / 28 = 28 признаков в подвекторе. Train использует только общий X_train_f32.
Train и add измеряются отдельно; Build/Train Time = Train Time + Add Time строго по заданию.
Создание оболочки индекса и настройки входят в RSS, но не в эту сумму времени.
Seed=42 задаётся как coarse clustering, так и PQ clustering; остальные настройки библиотечные.

Предупреждения FAISS о количестве training points не скрываются. Если FAISS позволяет обучение,
эксперимент продолжается без изменения train, nlist, m, bits или nprobe.
Результаты search — приближённые квадраты L2 и локальные индексы train; значение -1
считается ошибкой, не подменяется случайным соседом.

In [ ]:
def build_ivfpq():
    assert D % M_SUBQUANTIZERS == 0, "Несовместимая размерность PQ."
    quantizer = faiss.IndexFlatL2(D)
    index = faiss.IndexIVFPQ(quantizer, D, NLIST, M_SUBQUANTIZERS,
                            BITS_PER_CODE, faiss.METRIC_L2)
    index.cp.seed = RANDOM_STATE
    index.pq.cp.seed = RANDOM_STATE
    _, train_seconds = timed_call(index.train, X_train_f32)
    if not index.is_trained:
        raise ValueError("Обучение IVFPQ не завершено.")
    _, add_seconds = timed_call(index.add, X_train_f32)
    index.nprobe = NPROBE
    if index.ntotal != N_TRAIN:
        raise ValueError("IVFPQ не содержит все train-объекты.")
    return index, {
        "Train Time (s)": train_seconds,
        "Add Time (s)": add_seconds,
        "Build/Train Time (s)": train_seconds + add_seconds,
    }


def search_ivfpq(index):
    return index.search(X_test_f32, K_MAX)


ivfpq_distances, ivfpq_indices, ivfpq_stats = benchmark(build_ivfpq, search_ivfpq)
validate_indices(ivfpq_indices, "IVFPQ")
save_neighbors("ivfpq_neighbors_k10.pkl", ivfpq_indices, ivfpq_distances,
               configuration["ivfpq"], ivfpq_stats)
all_indices["IVFPQ"] = ivfpq_indices
all_stats["IVFPQ"] = ivfpq_stats
print("IVFPQ retrieval завершён:", ivfpq_indices.shape)

## 13. Валидация результатов всех методов

Каждый метод должен вернуть 500 × 10 корректных train-индексов.
Recall рассчитывается только после окончания поиска, относительно exact_indices.
Ошибки shapes, диапазона или метрик останавливают выполнение.

In [ ]:
ALGORITHMS = ["Exact Brute Force", "FlyHash", "HNSW", "IVFPQ"]
recalls = {}
for name in ALGORITHMS:
    validate_indices(all_indices[name], name)
    recalls[name] = {}
    for k in (1, 5, 10):
        value = recall_at_k(exact_indices, all_indices[name], k)
        if not np.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f"{name}: некорректный Recall@{k}.")
        recalls[name][f"Recall@{k}"] = value
    for column, value in all_stats[name].items():
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"{name}: некорректное измерение {column}.")
for k in (1, 5, 10):
    assert np.isclose(recalls["Exact Brute Force"][f"Recall@{k}"], 1.0)
print("Индексы и диапазоны Recall проверены для всех четырёх методов.")

## 14. Примеры соседей

Для query 0, 1 и 2 показаны локальные индексы train и пересечения top-10 с Exact.
Это проверка retrieval, а не сравнение меток цифр.

In [ ]:
for query_id in (0, 1, 2):
    print(f"Запрос {query_id}, исходная строка CSV {test_original_indices[query_id]}:")
    for name in ALGORITHMS:
        row = all_indices[name][query_id]
        overlap = np.intersect1d(exact_indices[query_id], row).size
        print(f"  {name}: {row.tolist()}; пересечение top-10: {overlap}/10")
    print()

## 15. Итоговая таблица

Обязательные имена столбцов сохранены буквально по заданию.
Recall хранится числом 0..1, время и память — без преждевременного округления.
Дополнительные столбцы позволяют увидеть train/add IVFPQ и составляющие RSS.
Таблица сохраняется в results/ann_results.csv.

In [ ]:
required_columns = [
    "Algorithm", "Recall@1", "Recall@5", "Recall@10", "Build/Train Time (s)",
    "Search Time Total (s)", "Search Time per Query (ms)", "Peak Memory (MB)",
]
rows = [dict(Algorithm=name, **recalls[name], **all_stats[name]) for name in ALGORITHMS]
results_df = pd.DataFrame(rows)
results_df = results_df[required_columns + [c for c in results_df if c not in required_columns]]
results_df.to_csv(RESULTS_DIR / "ann_results.csv", index=False, float_format="%.17g")
display(results_df.style.format(precision=6, na_rep="—"))
metrics = results_df.set_index("Algorithm")

## 16. Recall@K в зависимости от K

Линии всех четырёх методов строятся из рассчитанных Recall@1/5/10.
Exact показывает совпадение ground truth с самим собой.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in ALGORITHMS:
    ax.plot([1, 5, 10], [metrics.loc[name, f"Recall@{k}"] for k in (1, 5, 10)],
            marker="o", label=name)
ax.set(title="Полнота поиска ближайших соседей MNIST", xlabel="Число соседей K", ylabel="Recall@K")
ax.set_xticks([1, 5, 10])
ax.set_ylim(-0.02, 1.04)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "recall_vs_k.png", dpi=180)
plt.show()
plt.close(fig)

## 17. Recall@10 и время поиска

Scatter использует среднее время одной query в миллисекундах и реальный Recall@10.
Для FlyHash время включает кодирование test. Каждая точка подписана.

In [ ]:
def plot_tradeoff(column, xlabel, title, filename):
    fig, ax = plt.subplots(figsize=(9, 5))
    for name in ALGORITHMS:
        x = float(metrics.loc[name, column])
        y_value = float(metrics.loc[name, "Recall@10"])
        ax.scatter(x, y_value, s=65)
        ax.annotate(name, (x, y_value), xytext=(6, 6), textcoords="offset points")
    ax.set(title=title, xlabel=xlabel, ylabel="Recall@10")
    ax.set_ylim(-0.02, 1.08)
    ax.margins(x=0.25)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=180)
    plt.show()
    plt.close(fig)


plot_tradeoff("Search Time per Query (ms)", "Среднее время поиска на запрос (мс)",
              "Recall@10 и время поиска", "recall10_vs_search_time.png")

## 18. Recall@10 и память

Scatter использует одинаковую для всех методов оценку дополнительного peak RSS.
Она не равна сериализованному размеру индекса.

In [ ]:
plot_tradeoff("Peak Memory (MB)", "Дополнительный пиковый RSS (МБ)",
              "Recall@10 и потребление памяти", "recall10_vs_memory.png")

## 19. Проверка H1

HNSW подтверждает гипотезу только при Recall@10 > 0.95 **и** среднем времени query < 10 мс.
Использованы исходные ef=50 и ef_construction=200.

In [ ]:
hypotheses = {}
def show_hypothesis(name, condition):
    hypotheses[name] = bool(condition)
    print(f"{name} — " + ("ПОДТВЕРЖДЕНА" if condition else "НЕ ПОДТВЕРЖДЕНА"))

hnsw_recall10 = metrics.loc["HNSW", "Recall@10"]
hnsw_query_ms = metrics.loc["HNSW", "Search Time per Query (ms)"]
show_hypothesis("H1", hnsw_recall10 > 0.95 and hnsw_query_ms < 10)
print(f"Фактически: Recall@10={hnsw_recall10:.6f}; время запроса={hnsw_query_ms:.6f} мс.")

## 20. Проверка H2

PQ в формулировке означает IVFPQ. FlyHash должен иметь строго меньший дополнительный peak RSS,
чем оба других ANN-метода. При равенстве гипотеза не подтверждается.

In [ ]:
ANN_METHODS = ["FlyHash", "HNSW", "IVFPQ"]
ann_memory = metrics.loc[ANN_METHODS, "Peak Memory (MB)"]
show_hypothesis("H2", ann_memory["FlyHash"] < ann_memory["HNSW"]
                and ann_memory["FlyHash"] < ann_memory["IVFPQ"])
display(ann_memory.rename("Дополнительный пиковый RSS (МБ)").to_frame())

## 21. Проверка H3

Memory Efficiency = Recall@10 / PeakMemoryMB — воспроизводимая **операционализация**
понятия «компромисс точность–память», а не единственно возможное математическое определение.
Сравниваются только FlyHash, HNSW, IVFPQ. Если IVFPQ разделяет максимум с другим методом,
условие «имеет максимальный» выполнено; уникальность максимума отдельно не требуется.

При нулевом наблюдаемом приросте RSS отношение не определено. Оно сохраняется как NaN,
деление на искусственный epsilon не используется; H3 в таком запуске считается
не подтверждённой из-за недостаточности измерения. Метод с нулём нельзя объявить бесконечно
эффективным. Для итогового вывода максимум определяется среди корректных отношений,
с явной оговоркой, если сравнение всех ANN невозможно.

In [ ]:
ann_recall = metrics.loc[ANN_METHODS, "Recall@10"]
memory_efficiency = ann_recall / ann_memory.where(ann_memory > 0)
efficiency_defined = bool(memory_efficiency.notna().all())
if not efficiency_defined:
    warnings.warn("У ANN-метода нулевой наблюдаемый прирост RSS; H3 полностью оценить нельзя.")
h3_condition = (efficiency_defined and memory_efficiency["IVFPQ"] == memory_efficiency.max())
show_hypothesis("H3", h3_condition)
display(memory_efficiency.rename("Memory Efficiency (Recall@10 / МБ)").to_frame())
results_df["Memory Efficiency (Recall@10 / MB)"] = (
    results_df["Recall@10"] / results_df["Peak Memory (MB)"].where(results_df["Peak Memory (MB)"] > 0)
)
results_df.to_csv(RESULTS_DIR / "ann_results.csv", index=False, float_format="%.17g")

## 22. Проверка H4

Каждый из трёх ANN должен одновременно быть быстрее Exact по среднему query time
и иметь Recall@10 > 0.80. Одного несоответствующего метода достаточно для неподтверждения H4.

In [ ]:
exact_query_ms = metrics.loc["Exact Brute Force", "Search Time per Query (ms)"]
h4_rows = []
for name in ANN_METHODS:
    recall10 = metrics.loc[name, "Recall@10"]
    query_ms = metrics.loc[name, "Search Time per Query (ms)"]
    h4_rows.append({
        "Алгоритм": name, "Recall@10": recall10,
        "Время ANN на запрос (мс)": query_ms, "Время Exact на запрос (мс)": exact_query_ms,
        "Быстрее Exact": bool(query_ms < exact_query_ms), "Recall > 0.8": bool(recall10 > 0.80),
    })
h4_df = pd.DataFrame(h4_rows)
show_hypothesis("H4", (h4_df["Быстрее Exact"] & h4_df["Recall > 0.8"]).all())
display(h4_df)

## 23. Автоматические итоговые выводы

Выводы формируются исключительно из измерений текущего выполнения.
При равных значениях перечисляются все методы с экстремумом.
H1–H4 — проверки заданных порогов одного эксперимента, не статистические тесты значимости.
Никакие параметры не подстраиваются для их подтверждения.

In [ ]:
def extremum_text(series, maximize=True):
    valid = series.dropna()
    if valid.empty:
        return "не определён: нет корректных значений"
    value = valid.max() if maximize else valid.min()
    winners = ", ".join(valid.index[valid == value])
    return f"{winners} ({value:.6g})"

ann_metrics = metrics.loc[ANN_METHODS]
confirmed = ", ".join(name for name, passed in hypotheses.items() if passed) or "нет"
not_confirmed = ", ".join(name for name, passed in hypotheses.items() if not passed) or "нет"
summary = (
    "### Итоги измерений\n\n"
    f"- Максимальный Recall@10 среди ANN: {extremum_text(ann_metrics['Recall@10'])}.\n"
    f"- Минимальное время запроса среди ANN, мс: "
    f"{extremum_text(ann_metrics['Search Time per Query (ms)'], False)}.\n"
    f"- Минимальный дополнительный peak RSS среди ANN, МБ: "
    f"{extremum_text(ann_metrics['Peak Memory (MB)'], False)}.\n"
    f"- Максимальная Memory Efficiency: {extremum_text(memory_efficiency)}.\n"
    f"- Подтверждены: {confirmed}.\n"
    f"- Не подтверждены: {not_confirmed}.\n"
)
if not efficiency_defined:
    summary += "\nСравнение Memory Efficiency всех ANN невозможно из-за нулевого прироста RSS.\n"
summary += (
    "\nВыводы относятся к данному split и окружению. RSS зависит от allocator и порядка методов; "
    "измеренные время и память являются экспериментальными оценками."
)
display(Markdown(summary))
(RESULTS_DIR / "summary.md").write_text(summary, encoding="utf-8")
with (RESULTS_DIR / "hypotheses.json").open("w", encoding="utf-8") as stream:
    json.dump({"confirmed": hypotheses, "h3_all_efficiencies_defined": efficiency_defined},
              stream, ensure_ascii=False, indent=2)

## 24. Сохранённые файлы и финальная самопроверка

Проверяются артефакты текущего последовательного запуска: четыре pickle, CSV и три графика.
Дополнительно сохранены исходные split indices, манифест окружения/конфигурации,
автоматические выводы и статусы гипотез. Большие hash-коды и индексы поиска не сохраняются.
Ниже проверяется неизменность общих train/test и фактические параметры.

In [ ]:
assert hashlib.sha256(X_train_f32.tobytes()).hexdigest() == manifest["train_sha256"]
assert hashlib.sha256(X_test_f32.tobytes()).hexdigest() == manifest["test_sha256"]
assert HASH_DIMENSION == 7840 and SPARSITY_RATIO == 0.05
assert EF_CONSTRUCTION == 200 and EF == 50 and HNSW_M == 16
assert (NLIST, M_SUBQUANTIZERS, BITS_PER_CODE, NPROBE) == (100, 28, 8, 10)
assert D % M_SUBQUANTIZERS == 0
assert list(results_df["Algorithm"]) == ALGORITHMS
assert set(hypotheses) == {"H1", "H2", "H3", "H4"}
assert np.isclose(all_stats["IVFPQ"]["Build/Train Time (s)"],
                  all_stats["IVFPQ"]["Train Time (s)"] + all_stats["IVFPQ"]["Add Time (s)"])
for name in ALGORITHMS:
    assert np.isclose(all_stats[name]["Search Time per Query (ms)"],
                      all_stats[name]["Search Time Total (s)"] / 500 * 1000)

expected_files = [
    "exact_neighbors_k10.pkl", "flyhash_neighbors_k10.pkl", "hnsw_neighbors_k10.pkl",
    "ivfpq_neighbors_k10.pkl", "ann_results.csv", "recall_vs_k.png",
    "recall10_vs_search_time.png", "recall10_vs_memory.png",
    "split_indices.npz", "experiment_manifest.json", "summary.md", "hypotheses.json",
]
for filename in expected_files:
    path = RESULTS_DIR / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Ожидаемый артефакт отсутствует или пуст: {path}")
    print(f"{path} — {path.stat().st_size:,} байт")

saved_table = pd.read_csv(RESULTS_DIR / "ann_results.csv")
assert set(required_columns).issubset(saved_table.columns)
assert list(saved_table["Algorithm"]) == ALGORITHMS
for column in required_columns[1:]:
    assert np.allclose(saved_table[column], results_df[column], rtol=1e-12, atol=1e-12)
print("Самопроверка завершена. Все результаты относятся к реальному выполнению этих ячеек.")